In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from pathlib import Path


from sklearn.feature_selection import mutual_info_classif
from sklearn.utils.class_weight import compute_class_weight

import warnings
warnings.filterwarnings("ignore")

In [2]:
PROJECT_ROOT = Path(".").resolve().parent 

DATA_DIR   = PROJECT_ROOT / "Data"  
ASSET_DIR  = PROJECT_ROOT / "Assets"   

df_2015 = pd.read_csv(DATA_DIR / 'diabetes_012_health_indicators_BRFSS2015.csv')
df_2021 = pd.read_csv(DATA_DIR / 'diabetes_012_health_indicators_BRFSS2021.csv')

In [3]:
df_2015['Year'] = 2015
df_2021['Year'] = 2021

In [4]:
df_2015.Diabetes_012.value_counts()

Diabetes_012
0.0    213703
2.0     35346
1.0      4631
Name: count, dtype: int64

In [5]:
df_2021.Diabetes_012.value_counts()

Diabetes_012
0.0    197191
2.0     33568
1.0      5619
Name: count, dtype: int64

# just combined

In [6]:
combined_df = pd.concat([df_2015, df_2021], axis=0, ignore_index=True)
print(combined_df.shape)

(490058, 23)


In [7]:
print("duplicates  : ", combined_df.duplicated().sum(), "\n")
combined_df = combined_df.drop_duplicates()

# Create the binary target column: 0 = Healthy, 1 = At Risk (Prediabetes + Diabetes)
combined_df['Health_Status'] = (combined_df['Diabetes_012'] > 0).astype(int)

# Add a text-based label column for visual clarity if needed
combined_df['Health_Status_Label'] = combined_df['Health_Status'].map({0: 'Healthy', 1: 'At Risk'})

print("class value count : ", combined_df.Diabetes_012.value_counts(), "\n")
print(f"Combined Shape: {combined_df.shape}")
print("\nBinary Class Distribution in Overall Dataset:")
print(combined_df['Health_Status'].value_counts())

duplicates  :  36727 

class value count :  Diabetes_012
0.0    374597
2.0     68492
1.0     10242
Name: count, dtype: int64 

Combined Shape: (453331, 25)

Binary Class Distribution in Overall Dataset:
Health_Status
0    374597
1     78734
Name: count, dtype: int64


In [8]:
combined_df.head()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Year,Health_Status,Health_Status_Label
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,18.0,15.0,1.0,0.0,9.0,4.0,3.0,2015,0,Healthy
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,7.0,6.0,1.0,2015,0,Healthy
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,30.0,30.0,1.0,0.0,9.0,4.0,8.0,2015,0,Healthy
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,11.0,3.0,6.0,2015,0,Healthy
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,3.0,0.0,0.0,0.0,11.0,5.0,4.0,2015,0,Healthy


# balanced

In [9]:
df_healthy = combined_df[combined_df['Health_Status'] == 0]
df_at_risk = combined_df[combined_df['Health_Status'] == 1]

n_at_risk = len(df_at_risk)

df_healthy_downsampled = df_healthy.sample(n=n_at_risk, random_state=42)

balanced_df = pd.concat([df_healthy_downsampled, df_at_risk], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced Binary Class Distribution:")
print(balanced_df['Health_Status'].value_counts())
print(f"Balanced Dataset Shape: {balanced_df.shape}")

Balanced Binary Class Distribution:
Health_Status
0    78734
1    78734
Name: count, dtype: int64
Balanced Dataset Shape: (157468, 25)


In [10]:
combined_df.to_csv(DATA_DIR / 'diabetes_012_health_indicators_BRFSS2015_BRFSS2021.csv', index=False)
# balanced_df.to_csv(DATA_DIR / 'balanced_diabetes_012_health_indicators_BRFSS2015_BRFSS2021.csv', index=False)

print(f"2015 Shape: {df_2015.shape}")
print(f"2021 Shape: {df_2021.shape}")
print(f"Combined Shape: {combined_df.shape}")
# print(f"Balanced Shape: {balanced_df.shape}")

2015 Shape: (253680, 23)
2021 Shape: (236378, 23)
Combined Shape: (453331, 25)
